In [5]:
import sys
!{sys.executable} -m pip install interpax

In [ ]:
import os
import numpy as np
import math
import matplotlib.pyplot as plt
import copy
import corner
import json
import pickle
import glob

import astropy.units as u
import astropy.constants as astropy_const
import astropy.cosmology.units as cu
from astropy.cosmology import LambdaCDM
import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
from jax import jit
from jax.experimental.ode import odeint
from jax.lax import stop_gradient
import jax.scipy.special as jsp
from jax import vmap

from jax.scipy.stats import norm
import re
import h5py

import discovery as ds
import deterministic_eccentric as det_ecc

import jexplore.tools.distributions as d
from jexplore.backends import DefaultBackend
from jexplore.sampler import JaxSampler, Steps
from jexplore.sampling import EpochMH, SamplingMH
from jexplore.steps import Stretch, TSwap, DEStep


day = 24*3600
yr = 365.25*day
f_yr = 1/yr

Msun = astropy_const.M_sun.to(u.kg).value
c = astropy_const.c.to(u.m / u.s).value
G = astropy_const.G.to(u.m**3 / (u.kg * u.s**2)).value
MGsunsec = Msun*G / c**3
kpc = 1.0* astropy_const.kpc.to(u.m).value/c


ModuleNotFoundError: No module named 'discovery'

In [10]:
dirpath = '/home/smanzini/ptauser/work/WN+CRN+CGW_e0.75_3nHz_feathers_realWN_SNR8_seed290699/'
with open(dirpath + 'injected_params.json', 'r') as f:
    injected_dict = json.load(f)

with open(dirpath + 'prior_dict.json', 'r') as f:
    priors_all = json.load(f)

print(injected_dict)


{'J0030+0451_cw_psrdist': 0.28, 'J0030+0451_cw_xi0p': 1.9163898891237885, 'J0613-0200_cw_psrdist': 0.9, 'J0613-0200_cw_xi0p': 1.9348490455536158, 'J0751+1807_cw_psrdist': 1.0, 'J0751+1807_cw_xi0p': 0.0962933399642707, 'J0900-3144_cw_psrdist': 1.0, 'J0900-3144_cw_xi0p': -1.3458496214483335, 'J1012+5307_cw_psrdist': 0.7, 'J1012+5307_cw_xi0p': -2.8027360567794943, 'J1022+1001_cw_psrdist': 0.52, 'J1022+1001_cw_xi0p': -0.7328149346083426, 'J1024-0719_cw_psrdist': 0.49, 'J1024-0719_cw_xi0p': -0.5750798109183086, 'J1455-3330_cw_psrdist': 0.74, 'J1455-3330_cw_xi0p': -2.857120220482243, 'J1600-3053_cw_psrdist': 2.4, 'J1600-3053_cw_xi0p': -2.835238921937138, 'J1640+2224_cw_psrdist': 1.19, 'J1640+2224_cw_xi0p': 3.1364160318718435, 'J1713+0747_cw_psrdist': 1.05, 'J1713+0747_cw_xi0p': 0.9573633631976515, 'J1730-2304_cw_psrdist': 0.51, 'J1730-2304_cw_xi0p': -1.6681216000742336, 'J1738+0333_cw_psrdist': 1.0, 'J1738+0333_cw_xi0p': -0.40873658405505475, 'J1744-1134_cw_psrdist': 0.42, 'J1744-1134_cw_xi0

In [ ]:
feathers_path = 'EPTA_feather_DR2new/'

s_dsfiles = np.sort(os.listdir(feathers_path))
print(s_dsfiles)
d_psrs = [ds.pulsar.Pulsar.read_feather(feathers_path + f'{psrfile}') for psrfile in s_dsfiles]
print(d_psrs)
timedelay = det_ecc.make_delay_eccentric()
cw_common = ['cw_cos_gwtheta', 'cw_gwphi', 'cw_log10_M', 'cw_nu',
             'cw_log10_dist', 'cw_log10_Forb', 'cw_cos_inc', 'cw_psi', 'cw_gamma0',
             'cw_xi0','cw_e0']

def injection_EPTA_DR2new(psrs, crn_components = 30):

    pslmodels = []
    tspan = ds.getspan(psrs)
    for p in psrs:
        tspan = ds.getspan(p)

        model = [p.residuals, ds.makenoise_measurement(p, p.noisedict, tnequad = True), ds.makegp_timing(p, svd=True, variance =1e-40), ds.makedelay(p, timedelay, common=cw_common, name='cw')]
        
        model.append(ds.makegp_fourier(p, ds.powerlaw, crn_components, T=tspan, name='crn', common=['crn_log10_A', 'crn_gamma']))
        pslmodels.append(ds.PulsarLikelihood(model))

    tspan = ds.getspan(psrs)
    t0 = ds.getstart(psrs)
    return ds.GlobalLikelihood(psls = pslmodels)

injection = injection_EPTA_DR2new(d_psrs)
print(injection.logL.params)

key = jax.random.PRNGKey(290699)
key, injected_res = injection.sample(key, injected_dict)

NameError: name 'np' is not defined

In [ ]:
for i in range(len(d_psrs)):
    plt.plot(d_psrs[i].toas, injected_res[i])
    
    plt.show()

#  save in the psrs residuals
for ii, psr in enumerate(d_psrs):
    psr.residuals = np.array(injected_res[ii]).squeeze()

def fit_EPTA_DR2new(psrs, crn_components = 30):

    pslmodels = []
    tspan = ds.getspan(psrs)
    for p in psrs:
        tspan = ds.getspan(p)
        model = [p.residuals, ds.makenoise_measurement(p, p.noisedict, tnequad = True), ds.makegp_timing(p, svd=True)]

        model.append(ds.makegp_fourier(p, ds.powerlaw, crn_components, T=tspan, name='crn', common=['crn_log10_A', 'crn_gamma']))
        pslmodels.append(ds.PulsarLikelihood(model))

    tspan = ds.getspan(psrs)
    t0 = ds.getstart(psrs)
    return ds.GlobalLikelihood(psls = pslmodels)


crn_dict = {
    'crn_gamma': injected_dict['crn_gamma'],
    'crn_log10_A': injected_dict['crn_log10_A']
}

gl = EPTA_DR2new_fit_crn(d_psrs)
params_to_sample = gl.logL.params
jlogl = jax.jit(gl.logL)

# Warm up (Compiles)
_ = jlogl(crn_dict)

In [14]:
n_params = len(params_to_sample)

# Separate normal and uniform parameters
normal_indices = []
uniform_indices = []
normal_mus = []
normal_sigmas = []
uniform_lbs = []
uniform_ubs = []

for i, name in enumerate(params_to_sample):
    prior = priors_all[name]
    if prior['dist'] == 'normal':
        normal_indices.append(i)
        normal_mus.append(prior['mu'])
        normal_sigmas.append(prior['sigma'])
    else:  # uniform
        uniform_indices.append(i)
        uniform_lbs.append(prior['min'])
        uniform_ubs.append(prior['max'])

print(f"Normal indices: {normal_indices}")
print(f"Uniform indices: {uniform_indices}")
print(f"Total params: {n_params}")
print(f"Normal + Uniform = {len(normal_indices) + len(uniform_indices)}")

normal_mus = jnp.array(normal_mus)
normal_sigmas = jnp.array(normal_sigmas)
uniform_lbs = jnp.array(uniform_lbs)
uniform_ubs = jnp.array(uniform_ubs)
eps = 1e-10

normal_indices = jnp.array(normal_indices)
uniform_indices = jnp.array(uniform_indices)



@jax.jit
def prior_transform_single(u):
    """Transform [0,1]^n -> physical parameters"""
    x = jnp.zeros(n_params)

    # Transform normal parameters (extract subset, transform, assign back)
    if len(normal_indices) > 0:
        u_normal = u[normal_indices]  # Shape: (n_normal,)
        x_normal = normal_mus + jnp.maximum(normal_sigmas, eps) * ndtri(u_normal)
        x = x.at[normal_indices].set(x_normal)

    # Transform uniform parameters (extract subset, transform, assign back)
    if len(uniform_indices) > 0:
        u_uniform = u[uniform_indices]  # Shape: (n_uniform,)
        x_uniform = uniform_lbs + u_uniform * (uniform_ubs - uniform_lbs)
        x = x.at[uniform_indices].set(x_uniform)

    return x



@jax.jit
def physical_to_unit(x):
    """Transform physical parameters -> [0,1]^n hypercube"""
    u = jnp.zeros(n_params)
    if len(normal_indices) > 0:
        x_normal = x[normal_indices]
        u_normal = ndtr((x_normal - normal_mus) / jnp.maximum(normal_sigmas, eps))
        u = u.at[normal_indices].set(u_normal)
    if len(uniform_indices) > 0:
        x_uniform = x[uniform_indices]
        u_uniform = (x_uniform - uniform_lbs) / (uniform_ubs - uniform_lbs)
        u = u.at[uniform_indices].set(u_uniform)
    return u

@jax.jit
def loglike_single(u):
    x = prior_transform_single(u)
    x_dict = dict(zip(params_to_sample, x))
    logL = jlogl(x_dict)
    logL = jnp.where(jnp.isfinite(logL), logL, -1e30)
    return logL


NameError: name 'params_to_sample' is not defined

In [ ]:
ndim = n_params
nwalker = 50
temps = jnp.linspace(1, 3, 2)
ntemps = len(temps)

SamplingMH = SamplingMH(dim=ndim, nwalker=nwalker, temps=temps, loglik= loglike_single, logprior=d.Uniform(dim=ndim, minval = 0, maxval =1).leval)
Steps = Steps([{TSwap(permute = True).builder: 1.0}, {Stretch(permute = True).builder: 0.3}, {DEStep(permute = True).builder: 0.7}])
DefaultBackend = DefaultBackend(burn=0)


In [ ]:
key, subkey = jax.random.split(key)
initial_points = jax.random.uniform(subkey, shape=(ntemps * nwalker, ndim), minval=1e-6, maxval=1-1e-6)
print(initial_points.shape)  # (ntemps*nwalker, ndim)

iepoch = EpochMH({"p": initial_points})

sampler = JaxSampler(SamplingMH, Steps, DefaultBackend)

fepoch = sampler.run(iepoch, niters=100, nepoch=1, seed=13)
